# 03 – Lung masks, Grad-CAM and the lung-attention metric (Stage 3)

Stage 3 of P03 (Aman, issues #10–#14). This notebook runs the whole stage in order:

1. **Self-checks** on synthetic data (`src/explain/check_explain.py`): Grad-CAM, the metric and the mask format.
2. **Stage 3.2 – lung masks** for all 5,856 images (`src/masking/lung_masks.py`), plus figures for the eye check.
3. **Stage 3.3 – Grad-CAM + attention metric** for the custom CNN and DenseNet-121 on the test set
   (`src/explain/run_gradcam.py`), plus the figures for the shortcut analysis (Stage 3.4).

**How to run on Kaggle**
- Attach the dataset *Chest X-Ray Images (Pneumonia)* (Add Input → search `chest-xray-pneumonia`).
- Settings: **Internet on** (clone the repo, install `torchxrayvision`, download its PSPNet weights) and **GPU on** (T4).
- Make the two checkpoints available (section 2). Then Run All. Masks take a few minutes on a T4, Grad-CAM about one.

Outputs: `results/stage3/` and `results/stage3_metrics.csv` (download and commit them on `stage3-gradcam`),
and `/kaggle/working/lung_masks.zip` (upload to the team Drive folder; it is not committed).

In [ ]:
# Make the repo's code importable (same pattern as notebooks/01_eda.ipynb).
# On Kaggle the notebook runs in /kaggle/working, so we clone the repo (needs Internet on) and cd into it.
# Locally the notebook is opened from notebooks/, so we go one folder up to the repo root.
import os, sys, subprocess

REPO_URL = "https://github.com/Abhishek-Singh-2008/Pneumonia-detection.git"
BRANCH = "stage3-gradcam"   # switch to "main" once Stage 3 is merged

if os.path.isdir("/kaggle/working"):
    os.chdir("/kaggle/working")
    if not os.path.isdir("Pneumonia-detection"):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL], check=True)
    os.chdir("Pneumonia-detection")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Repo root:", os.getcwd())

In [ ]:
# torchxrayvision provides the pretrained lung segmentation model (PSPNet). It is not preinstalled on Kaggle.
# The version is pinned in requirements.txt; installing it does not change torch/torchvision.
!pip install -q torchxrayvision==1.5.7
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (turn GPU on)")

## 1. Self-checks (no data needed)
Grad-CAM must find a known answer on a toy model for both predicted classes (+logit for PNEUMONIA, −logit for NORMAL,
as agreed in #21), work on both real architectures, and the metric and mask files must behave as documented.

In [ ]:
# Every line must print PASS; an AssertionError means something is wrong and the rest should not be trusted.
!python -m src.explain.check_explain

## 2. Checkpoints of the two models
The weights are not in the repo (team Drive folder, see the README and the handover notes on #1 and #6).
This cell uses what is already there and only downloads what is missing:

1. It searches everything attached under `/kaggle/input` (e.g. a private Kaggle dataset with the two `.pth` files)
   and `/kaggle/working/checkpoints` for `custom_cnn_best.pth` and `densenet121_phaseB_lr1e-4_bs16_best.pth`.
2. Any file it does not find is downloaded from the team Google Drive with `gdown` into `/kaggle/working/checkpoints`
   (works while the Drive links stay "anyone with the link can view").

In [ ]:
from pathlib import Path
from src.explain.common import CHECKPOINT_FILES, MODELS, find_checkpoint

CKPT_DIR = Path("/kaggle/working/checkpoints") if os.path.isdir("/kaggle/working") else Path("checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH = [Path("/kaggle/input"), CKPT_DIR]   # places to look before downloading anything

# Google Drive sources from the handover notes: Stage 1 team folder (#1) and the Stage 2 final model file (#6).
DRIVE = {
    "custom_cnn": ("folder", "https://drive.google.com/drive/folders/1-IldsdQnswOB-PVVcwLyYlijN_4TlTYA"),
    "densenet121": ("file", "https://drive.google.com/uc?id=1RKlw15zgKJSDFrKXs0MrVCWqf9_Gv7MR"),
}

def locate(model_name):
    """Return the path of a model's checkpoint if it exists in any SEARCH folder, else None."""
    for root in SEARCH:
        if root.is_dir():
            try:
                return find_checkpoint(model_name, root)
            except FileNotFoundError:
                pass
    return None

for m in MODELS:
    found = locate(m)
    if found is None:
        # Not attached: download it from the team Drive into CKPT_DIR (searched again afterwards).
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        kind, url = DRIVE[m]
        if kind == "folder":
            subprocess.run(["gdown", "--folder", url, "-O", str(CKPT_DIR / "drive_folder")], check=True)
        else:
            subprocess.run(["gdown", url, "-O", str(CKPT_DIR / CHECKPOINT_FILES[m])], check=True)
        found = locate(m)
    if found is None:
        raise FileNotFoundError(f"{CHECKPOINT_FILES[m]} is neither attached nor downloadable; attach it as a Kaggle dataset.")
    # Copy (link) everything into CKPT_DIR so the scripts below get one folder with both files.
    target = CKPT_DIR / CHECKPOINT_FILES[m]
    if not target.exists():
        target.symlink_to(found)
    print(f"{m}: {found}")

## 3. Stage 3.2 – lung masks for every image
PSPNet (torchxrayvision, trained on adult X-rays) segments the left and right lung; they are joined, resized to
224×224 with nearest-neighbour, and saved as `{image_id}.png` (0 = background, 255 = lung, #21 row 14).
The whole image is resized without cropping, exactly like the classifier input, so masks line up with the heatmaps.
Masks that already exist are skipped, so the cell can be re-run after an interruption.

In [ ]:
MASK_DIR = Path("/kaggle/working/lung_masks") if os.path.isdir("/kaggle/working") else Path("data/lung_masks")
!python -m src.masking.lung_masks --out-dir {MASK_DIR} --zip

### Eye check (issue #12 task "Check a sample of masks by eye")
Look at both grids. A good mask covers both lung fields and little else. Write down what you see (how many look
right, typical failures such as a missed lower lobe, the heart included, or masks on very small children) in the
Stage 3 handover note and the report. `results/stage3/lung_mask_summary.csv` lists every image with its lung
area and a `flag` for implausible masks.

In [ ]:
from IPython.display import Image as Show, display
import pandas as pd

summary = pd.read_csv("results/stage3/lung_mask_summary.csv")
print(summary.groupby(["split", "label"])["lung_area"].describe().round(3))
print("Flagged:", int(summary["flag"].sum()), "of", len(summary))
display(Show("results/stage3/lung_masks_random.png"))
display(Show("results/stage3/lung_masks_flagged.png"))

## 4. Stage 3.3 – Grad-CAM and the lung-attention metric
For both models and every test image: Grad-CAM of the **predicted** class, then
`attention_in_lung = heatmap inside lung mask / whole heatmap`, its chance level `lung_area`, `lift = attention / area`,
and `border_fraction` (outer 10% band, where markers sit). The script first checks that its recomputed test metrics
equal the committed Stage 1/2 results (proves the right weights were loaded). Nothing is chosen on the test set.

In [ ]:
!python -m src.explain.run_gradcam --checkpoint-dir {CKPT_DIR} --mask-dir {MASK_DIR}

In [ ]:
# Main numbers: average attention per model and group, next to the chance level (lung_area_mean).
s = pd.read_csv("results/stage3/attention_summary.csv")
display(s[s["group"].isin(["all", "NORMAL", "PNEUMONIA", "correct", "wrong", "NORMAL_wrong"])])
display(pd.read_csv("results/stage3_metrics.csv"))
display(Show("results/stage3/attention_boxplot.png"))

## 5. Figures for the shortcut analysis (Stage 3.4)
The fixed 50-image sample (same images for both models, `results/stage3/sample_images.csv`) and three galleries
per model: least attention on the lungs, most attention on the border, and the most confident false positives.
Look for heatmaps on text markers ("R"/"L"), corners, borders, devices or labels, and pick 6–8 clear examples
for the report (issue #14).

In [ ]:
for p in sorted(Path("results/stage3").glob("heatmaps_sample_p*.png")):
    display(Show(str(p)))
for m in MODELS:
    for kind in ("lowest_attention", "highest_border", "false_positives"):
        display(Show(f"results/stage3/{kind}_{m}.png"))

## 6. Save the outputs
- `stage3_outputs.zip` (results) → download from the Output panel, unzip into the repo, commit on `stage3-gradcam`.
- `lung_masks.zip` → download and upload to the team Drive folder, then put the link in the README (issue #12).

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/stage3_outputs", "zip", root_dir=".", base_dir="results") if os.path.isdir("/kaggle/working") else None
!ls -la /kaggle/working/*.zip 2>/dev/null || echo "not on Kaggle: outputs are already in results/ and data/lung_masks/"